# Retrieval-Augmented Generation with a Vector Store

This notebook demonstrates chunking, embeddings, retrieval, grounded answering, and visualization with a deterministic local fixture. Optional live adapters use environment credentials only; offline execution requires no Azure or OpenAI account.

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics.pairwise import cosine_similarity
LIVE_OPENAI=os.getenv('RUN_LIVE_OPENAI','0')=='1'; LIVE_COSMOS=os.getenv('RUN_LIVE_COSMOS','0')=='1'
OUTPUT_DIR=Path('generated/openai-604'); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
documents=[
 {'id':'filing','text':'Zurich residents generally submit the annual tax return by March 31. An extension can be requested before the deadline.'},
 {'id':'payment','text':'Provisional tax invoices can be paid in installments. The final assessment reconciles the amount due.'},
 {'id':'deductions','text':'Eligible professional expenses and documented charitable donations may be declared as deductions.'},
 {'id':'support','text':'The cantonal tax office provides official forms and answers procedural questions.'}]

## 1. Embed and Retrieve Locally

In [ ]:
vectorizer=HashingVectorizer(n_features=128,alternate_sign=False,norm='l2')
doc_vectors=vectorizer.transform([item['text'] for item in documents]).toarray()
def retrieve(question,k=2):
    query_vector=vectorizer.transform([question]).toarray(); scores=cosine_similarity(query_vector,doc_vectors)[0]; order=np.argsort(scores)[::-1][:k]
    return pd.DataFrame([{'id':documents[i]['id'],'text':documents[i]['text'],'score':scores[i]} for i in order])
question='When should I submit my tax return, and can I request more time?'
results=retrieve(question); display(results)

## 2. Build a Grounded Answer

The offline answer is extractive and deterministic. Live generation uses only retrieved context. Production systems also need authorization filters, citation checks, prompt-injection defenses, telemetry, and evaluation.

In [ ]:
context='
'.join(results['text'])
if LIVE_OPENAI:
    from openai import OpenAI
    answer=OpenAI().responses.create(model=os.getenv('OPENAI_TEXT_MODEL','gpt-4.1-mini'),input=f'Answer only from this context and cite document ids.
{context}
Question: {question}').output_text.strip()
else:
    answer='The fixture states that the annual return is generally due March 31, and an extension can be requested before that deadline. [filing]'
print(answer)

## 3. Optional Cosmos DB for MongoDB vCore Adapter

In [ ]:
def cosmos_collection():
    if not LIVE_COSMOS: return None
    import pymongo
    connection=os.environ['COSMOS_MONGODB_CONNECTION_STRING']
    client=pymongo.MongoClient(connection,serverSelectionTimeoutMS=5000,tls=True)
    client.admin.command('ping')
    return client[os.getenv('COSMOS_DATABASE','rag_demo')][os.getenv('COSMOS_COLLECTION','chunks')]
print('Cosmos adapter:', 'enabled' if LIVE_COSMOS else 'disabled (offline default)')

## 4. Visualize the Embedding Fixture and Retrieval Scores

In [ ]:
points=PCA(n_components=2,random_state=604).fit_transform(doc_vectors)
figure,axes=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
axes[0].scatter(points[:,0],points[:,1],color='#2878b5');
for index,item in enumerate(documents): axes[0].annotate(item['id'],points[index])
axes[0].set(title='Local document embedding projection',xlabel='PCA 1',ylabel='PCA 2')
axes[1].bar(results['id'],results['score'],color='#d95f02'); axes[1].set(title='Top retrieval scores',ylabel='Cosine similarity',ylim=(0,1))
figure_path=OUTPUT_DIR/'offline-rag-retrieval.png'; figure.savefig(figure_path,dpi=170,bbox_inches='tight'); plt.show(); print('saved:',figure_path)